# Long-Only Constraint Analysis

NSE does not permit retail investors to carry short equity positions overnight. This notebook
re-runs the walk-forward backtest under that constraint: when the strategy signals a pair trade,
only the **long leg** is executed.

**What changes:**
- Signal = +1 (spread below lower threshold → A is cheap): buy A only (no short B)
- Signal = −1 (spread above upper threshold → A is expensive): buy B only (no short A)
- Market neutrality is lost — the portfolio now has directional exposure
- Transaction costs are halved (one leg instead of two)

The notebook compares three strategies:
1. **Full pairs** — theoretical market-neutral (both legs, requires futures for shorts)
2. **Long-only** — constrained to cash equity only
3. **Nifty 50** — buy-and-hold benchmark

In [1]:
import sys, os, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf

sys.path.insert(0, os.path.abspath(".."))
import config
from src.data_loader import load_data
from src.pair_selector import (
    generate_candidate_pairs, compute_spread, test_cointegration,
    test_cointegration_engle_granger, compute_half_life,
    test_cointegration_stability, score_pair,
)
from src.signal_generator import (
    compute_rolling_beta_fast, compute_rolling_spread,
    compute_rolling_zscore, apply_risk_overlay,
)
from src.backtester import compute_pair_returns, extract_trade_log
from src.metrics import compute_all_metrics

warnings.filterwarnings("ignore")

## Parameters and setup

In [2]:
TRAIN_WINDOW_DAYS = 504
REBALANCE_DAYS = 126
WARMUP_DAYS = 270
Z_ENTRY = 2.2
Z_EXIT = 0.5
Z_STOP = 3.5
VELOCITY_LOOKBACK = 2
PAIR_DD_STOP = 0.03
MIN_STAB = 60.0
MAX_PAIRS = 5
BETA_LOOKBACK = 120
Z_LOOKBACK = 30

## Core functions

In [3]:
def generate_signals_with_velocity(zscore, entry_thresh, exit_thresh, vel_lb):
    """Entry with velocity confirmation."""
    position = 0
    signals = np.zeros(len(zscore))
    zs_val = zscore.values
    for i in range(len(zs_val)):
        z = zs_val[i]
        if np.isnan(z):
            signals[i] = 0
            position = 0
            continue
        if position == 0:
            if z < -entry_thresh:
                if vel_lb > 0 and i >= vel_lb:
                    if zs_val[i] > zs_val[i - vel_lb]:
                        position = 1
                else:
                    position = 1
            elif z > entry_thresh:
                if vel_lb > 0 and i >= vel_lb:
                    if zs_val[i] < zs_val[i - vel_lb]:
                        position = -1
                else:
                    position = -1
        else:
            if abs(z) < exit_thresh:
                position = 0
        signals[i] = position
    return pd.Series(signals, index=zscore.index, name="signal")


def select_pairs(prices):
    candidates = generate_candidate_pairs(prices)
    results = []
    for pair in candidates:
        sa, sb, sec = pair["stock_a"], pair["stock_b"], pair["sector"]
        try:
            pa, pb = prices[sa], prices[sb]
            ols = compute_spread(pa, pb)
            adf = test_cointegration(ols["spread"])
            eg = test_cointegration_engle_granger(pa, pb)
            hl = compute_half_life(ols["spread"])
            stab = test_cointegration_stability(pa, pb)["stability_pct"] if adf["is_cointegrated"] else 0.0
            results.append({
                "stock_a": sa, "stock_b": sb, "sector": sec,
                "beta": ols["beta"], "adf_pvalue": adf["adf_pvalue"],
                "is_cointegrated": adf["is_cointegrated"],
                "eg_cointegrated": eg["eg_is_cointegrated"],
                "half_life": hl, "stability_pct": stab,
                "score": score_pair(adf["adf_pvalue"], hl, stab),
            })
        except Exception:
            continue
    if not results:
        return pd.DataFrame()
    df = pd.DataFrame(results).sort_values("score", ascending=False)
    mask = (df["is_cointegrated"] & df["eg_cointegrated"]
            & (df["half_life"] >= config.HALF_LIFE_MIN)
            & (df["half_life"] <= config.HALF_LIFE_MAX)
            & (df["stability_pct"] >= MIN_STAB))
    return df[mask].head(MAX_PAIRS)

## Long-only returns computation

The key modification: instead of `ret_A - β * ret_B`, we only take the return of whichever
stock the signal says to go long on.

In [4]:
def compute_long_only_returns(signals_df):
    """
    Compute returns using only the long leg of each pair trade.
    
    Signal = +1 → long A only (A is cheap relative to B)
    Signal = -1 → long B only (B is cheap relative to A)
    Signal =  0 → flat, no position
    """
    ret_a = np.log(signals_df["price_a"] / signals_df["price_a"].shift(1))
    ret_b = np.log(signals_df["price_b"] / signals_df["price_b"].shift(1))
    
    prev_signal = signals_df["signal"].shift(1).fillna(0)
    
    # Long A when signal was +1, long B when signal was -1
    daily_return = pd.Series(0.0, index=signals_df.index)
    daily_return[prev_signal == 1] = ret_a[prev_signal == 1]
    daily_return[prev_signal == -1] = ret_b[prev_signal == -1]
    
    # One leg only → half the transaction costs
    cost_per_trade = config.COST_PER_LEG + config.SLIPPAGE
    signal_change = signals_df["signal"].diff().abs().fillna(0)
    costs = signal_change * cost_per_trade
    
    return pd.DataFrame({
        "gross_return": daily_return,
        "cost": costs,
        "net_return": daily_return - costs,
    }, index=signals_df.index)

## Walk-forward engine (both strategies)

In [5]:
def run_period(prices, selected, pstart, pend):
    """Run a single walk-forward period for both full-pairs and long-only."""
    wstart = pstart - pd.Timedelta(days=WARMUP_DAYS)
    sp = prices.loc[wstart:pend]
    
    full_ret, long_ret = {}, {}
    
    for _, pair in selected.iterrows():
        sa, sb, hl = pair["stock_a"], pair["stock_b"], pair["half_life"]
        try:
            pa, pb = sp[sa], sp[sb]
        except KeyError:
            continue
        
        beta = compute_rolling_beta_fast(pa, pb, BETA_LOOKBACK)
        spread = compute_rolling_spread(pa, pb, beta)
        zs = compute_rolling_zscore(spread, Z_LOOKBACK)
        raw = generate_signals_with_velocity(zs, Z_ENTRY, Z_EXIT, VELOCITY_LOOKBACK)
        sig = apply_risk_overlay(raw, zs, hl, Z_STOP)
        
        sdf = pd.DataFrame({
            "price_a": pa, "price_b": pb, "beta": beta,
            "spread": spread, "zscore": zs, "signal": sig
        })
        sdf = sdf.loc[pstart:pend]
        if len(sdf) < 10:
            continue
        
        pk = f"{sa.replace('.NS','')}-{sb.replace('.NS','')}"
        
        # Full pairs returns (both legs)
        rdf_full = compute_pair_returns(sdf)
        cum = (1 + rdf_full["net_return"].fillna(0)).cumprod()
        dd = (cum - cum.cummax()) / cum.cummax()
        stops = dd[dd < -PAIR_DD_STOP].index
        if len(stops) > 0:
            rdf_full.loc[stops[0]:, ["net_return", "gross_return"]] = 0
        full_ret[pk] = rdf_full
        
        # Long-only returns (one leg)
        rdf_long = compute_long_only_returns(sdf)
        cum_lo = (1 + rdf_long["net_return"].fillna(0)).cumprod()
        dd_lo = (cum_lo - cum_lo.cummax()) / cum_lo.cummax()
        stops_lo = dd_lo[dd_lo < -PAIR_DD_STOP].index
        if len(stops_lo) > 0:
            rdf_long.loc[stops_lo[0]:, ["net_return", "gross_return"]] = 0
        long_ret[pk] = rdf_long
    
    return {"full": full_ret, "long_only": long_ret}

## Run walk-forward backtest

In [6]:
prices = load_data(force_refresh=False)
dates = prices.index
rb_indices = list(range(TRAIN_WINDOW_DAYS, len(dates), REBALANCE_DAYS))
print(f"Total walk-forward periods: {len(rb_indices)}")

full_all, long_all = [], []

for i, ri in enumerate(rb_indices):
    tp = prices.iloc[max(0, ri - TRAIN_WINDOW_DAYS):ri]
    te = min(ri + REBALANCE_DAYS, len(dates))
    ps, pe = dates[ri], dates[te - 1]
    
    sel = select_pairs(tp)
    if len(sel) == 0:
        print(f"  P{i+1}: {ps.date()} → {pe.date()} | No qualifying pairs")
        continue
    
    res = run_period(prices, sel, ps, pe)
    
    if res["full"]:
        full_rm = pd.DataFrame({n: r["net_return"] for n, r in res["full"].items()})
        full_all.append(full_rm.mean(axis=1).fillna(0))
    
    if res["long_only"]:
        long_rm = pd.DataFrame({n: r["net_return"] for n, r in res["long_only"].items()})
        long_all.append(long_rm.mean(axis=1).fillna(0))
    
    n_pairs = len(res["full"])
    print(f"  P{i+1}: {ps.date()} → {pe.date()} | {n_pairs} pairs")

# Combine returns
full_returns = pd.concat(full_all).sort_index()
full_returns = full_returns[~full_returns.index.duplicated(keep='first')]

long_returns = pd.concat(long_all).sort_index()
long_returns = long_returns[~long_returns.index.duplicated(keep='first')]

print(f"\nFull pairs: {len(full_returns)} trading days")
print(f"Long-only:  {len(long_returns)} trading days")

📂 Loading cached data from data/nse_prices.parquet

🧹 Cleaning data...
   Raw shape: (1315, 154)
   Clean shape: (1315, 154)
   Date range: 2021-01-01 to 2026-04-29
   Trading days: 1315
   ✅ No issues found

💾 Saved to data/nse_prices.parquet
Total walk-forward periods: 7
📊 Generated 774 candidate pairs across 15 sectors
  P1: 2023-01-12 → 2023-07-18 | 5 pairs
📊 Generated 774 candidate pairs across 15 sectors
  P2: 2023-07-19 → 2024-01-19 | 5 pairs
📊 Generated 774 candidate pairs across 15 sectors
  P3: 2024-01-23 → 2024-07-30 | 5 pairs
📊 Generated 774 candidate pairs across 15 sectors
  P4: 2024-07-31 → 2025-01-29 | 5 pairs
📊 Generated 774 candidate pairs across 15 sectors
  P5: 2025-01-30 → 2025-08-01 | 5 pairs
📊 Generated 774 candidate pairs across 15 sectors
  P6: 2025-08-04 → 2026-02-04 | 5 pairs
📊 Generated 774 candidate pairs across 15 sectors
  P7: 2026-02-05 → 2026-04-29 | 5 pairs

Full pairs: 811 trading days
Long-only:  811 trading days


## Download Nifty 50 benchmark

In [7]:
test_start = full_returns.index[0]
test_end = full_returns.index[-1]

nifty = yf.download(
    "^NSEI",
    start=test_start - pd.Timedelta(days=10),
    end=test_end + pd.Timedelta(days=1),
    auto_adjust=True,
    progress=False,
)
nifty_close = nifty["Close"].squeeze()
nifty_returns = nifty_close.pct_change().dropna()

common_idx = nifty_returns.index.intersection(full_returns.index)
nifty_returns = nifty_returns.loc[common_idx]
print(f"Nifty 50 benchmark: {len(nifty_returns)} days")

Nifty 50 benchmark: 810 days


## Performance comparison

In [8]:
full_metrics = compute_all_metrics(full_returns.dropna())
long_metrics = compute_all_metrics(long_returns.dropna())
nifty_metrics = compute_all_metrics(nifty_returns.dropna())

comparison = pd.DataFrame({
    "Full Pairs (theoretical)": {
        "Total Return (%)": round(full_metrics["total_return"], 2),
        "CAGR (%)": round(full_metrics["cagr"], 2),
        "Sharpe Ratio": round(full_metrics["sharpe_ratio"], 2),
        "Max Drawdown (%)": round(full_metrics["max_drawdown"], 2),
        "Volatility (%)": round(full_metrics["volatility"], 2),
        "Calmar Ratio": round(full_metrics["calmar_ratio"], 2),
    },
    "Long-Only (NSE constraint)": {
        "Total Return (%)": round(long_metrics["total_return"], 2),
        "CAGR (%)": round(long_metrics["cagr"], 2),
        "Sharpe Ratio": round(long_metrics["sharpe_ratio"], 2),
        "Max Drawdown (%)": round(long_metrics["max_drawdown"], 2),
        "Volatility (%)": round(long_metrics["volatility"], 2),
        "Calmar Ratio": round(long_metrics["calmar_ratio"], 2),
    },
    "Nifty 50 B&H": {
        "Total Return (%)": round(nifty_metrics["total_return"], 2),
        "CAGR (%)": round(nifty_metrics["cagr"], 2),
        "Sharpe Ratio": round(nifty_metrics["sharpe_ratio"], 2),
        "Max Drawdown (%)": round(nifty_metrics["max_drawdown"], 2),
        "Volatility (%)": round(nifty_metrics["volatility"], 2),
        "Calmar Ratio": round(nifty_metrics["calmar_ratio"], 2),
    },
})

comparison

,Full Pairs (theoretical),Long-Only (NSE constraint),Nifty 50 B&H
Total Return (%),42.90,44.70,35.10
CAGR (%),11.73,12.17,9.81
Sharpe Ratio,1.27,1.32,0.33
Max Drawdown (%),-1.35,-1.82,-15.77
Volatility (%),4.06,4.24,12.79
Calmar Ratio,8.71,6.69,0.62


## Equity curves

In [9]:
full_equity = (1 + full_returns).cumprod()
long_equity = (1 + long_returns).cumprod()
nifty_equity = (1 + nifty_returns).cumprod()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=full_equity.index, y=(full_equity - 1) * 100,
    name="Full Pairs (theoretical)",
    line=dict(color="#00e676", width=2.5)
))

fig.add_trace(go.Scatter(
    x=long_equity.index, y=(long_equity - 1) * 100,
    name="Long-Only (NSE constraint)",
    line=dict(color="#ff9100", width=2.5)
))

fig.add_trace(go.Scatter(
    x=nifty_equity.index, y=(nifty_equity - 1) * 100,
    name="Nifty 50 B&H",
    line=dict(color="#29b6f6", width=1.5, dash="dash")
))

fig.update_layout(
    title="Cumulative Returns: Full Pairs vs Long-Only vs Nifty 50",
    xaxis_title="Date", yaxis_title="Return (%)",
    template="plotly_dark", height=500,
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)
fig.show()

## Drawdown comparison

In [10]:
def drawdown_series(equity):
    peak = equity.cummax()
    return (equity - peak) / peak

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=full_equity.index, y=drawdown_series(full_equity) * 100,
    name="Full Pairs", fill="tozeroy",
    line=dict(color="#00e676", width=1.5)
))

fig.add_trace(go.Scatter(
    x=long_equity.index, y=drawdown_series(long_equity) * 100,
    name="Long-Only",
    line=dict(color="#ff9100", width=1.5)
))

fig.add_trace(go.Scatter(
    x=nifty_equity.index, y=drawdown_series(nifty_equity) * 100,
    name="Nifty 50 B&H",
    line=dict(color="#29b6f6", width=1, dash="dash")
))

fig.update_layout(
    title="Drawdown Comparison",
    xaxis_title="Date", yaxis_title="Drawdown (%)",
    template="plotly_dark", height=350,
    hovermode="x unified",
    legend=dict(yanchor="bottom", y=0.01, xanchor="left", x=0.01)
)
fig.show()

## Rolling 60-day Sharpe ratio

In [11]:
def rolling_sharpe(returns, window=60):
    rf = 0.06 / 252
    excess = returns - rf
    return np.sqrt(252) * excess.rolling(window).mean() / excess.rolling(window).std()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=full_returns.index, y=rolling_sharpe(full_returns),
    name="Full Pairs", line=dict(color="#00e676", width=1.5)
))

fig.add_trace(go.Scatter(
    x=long_returns.index, y=rolling_sharpe(long_returns),
    name="Long-Only", line=dict(color="#ff9100", width=1.5)
))

fig.add_hline(y=0, line_dash="dot", line_color="#8a94a6")

fig.update_layout(
    title="Rolling 60-Day Sharpe Ratio",
    xaxis_title="Date", yaxis_title="Sharpe Ratio",
    template="plotly_dark", height=350,
    hovermode="x unified"
)
fig.show()

## Market correlation analysis

The full-pairs strategy should have near-zero correlation with the market (market-neutral).
The long-only strategy will have higher market correlation since it carries directional exposure.

In [12]:
common = full_returns.index.intersection(nifty_returns.index)

full_corr = full_returns.loc[common].corr(nifty_returns.loc[common])
long_corr = long_returns.loc[common].corr(nifty_returns.loc[common])

full_beta = full_returns.loc[common].cov(nifty_returns.loc[common]) / nifty_returns.loc[common].var()
long_beta = long_returns.loc[common].cov(nifty_returns.loc[common]) / nifty_returns.loc[common].var()

market_stats = pd.DataFrame({
    "Full Pairs": {
        "Correlation with Nifty 50": round(full_corr, 3),
        "Market Beta": round(full_beta, 3),
    },
    "Long-Only": {
        "Correlation with Nifty 50": round(long_corr, 3),
        "Market Beta": round(long_beta, 3),
    },
})

market_stats

,Full Pairs,Long-Only
Correlation with Nifty 50,0.012,0.227
Market Beta,0.004,0.075
